# Notebook 2 — Train on SageMaker

This notebook:
1. Uploads CIFAR-10 data to S3
2. Launches a SageMaker **PyTorch Training Job** (ml.g4dn.xlarge)
3. Monitors training metrics in real-time
4. Downloads the best checkpoint

> **Prerequisites:** Complete `01_intro_pytorch.ipynb` and run `scripts/setup_aws.py`.

In [ ]:
import os, sys, json
sys.path.insert(0, '../src')

import boto3
import sagemaker
from sagemaker.pytorch import PyTorch
from dotenv import load_dotenv
from pathlib import Path

load_dotenv('../.env')

REGION       = os.environ['AWS_REGION']
BUCKET       = os.environ['S3_BUCKET']
ROLE_ARN     = os.environ['SAGEMAKER_ROLE_ARN']

session = sagemaker.Session(boto_session=boto3.Session(region_name=REGION))
print(f'SageMaker SDK : {sagemaker.__version__}')
print(f'S3 bucket     : {BUCKET}')
print(f'Role ARN      : {ROLE_ARN}')

## 1. Upload CIFAR-10 to S3

SageMaker copies data from S3 to the training instance's `/opt/ml/input/data/training` before the job starts.

In [ ]:
import torchvision
from pathlib import Path

# Download CIFAR-10 locally first
local_data = Path('../datasets/cifar10')
local_data.mkdir(parents=True, exist_ok=True)
torchvision.datasets.CIFAR10(root=str(local_data), train=True,  download=True)
torchvision.datasets.CIFAR10(root=str(local_data), train=False, download=True)
print('CIFAR-10 downloaded.')

# Upload to S3
s3_data_uri = session.upload_data(
    path=str(local_data),
    bucket=BUCKET,
    key_prefix='cifar10'
)
print(f'Data uploaded → {s3_data_uri}')

In [ ]:
# Data already uploaded to S3 via aws s3 sync — skip the upload above
s3_data_uri = f's3://{BUCKET}/cifar10'
print(f'Data URI: {s3_data_uri}')

## 2. Configure the SageMaker PyTorch Estimator

In [ ]:
estimator = PyTorch(
    entry_point='train.py',
    source_dir='../src',
    role=ROLE_ARN,
    framework_version='2.2',
    py_version='py311',
    instance_type='ml.g4dn.xlarge',   # 1x T4 GPU, 16 GB RAM
    instance_count=1,
    volume_size=30,                    # GB
    max_run=3600,                      # max 1 hour
    output_path=f's3://{BUCKET}/training-output',
    sagemaker_session=session,
    hyperparameters={
        'epochs':       30,
        'batch-size':   128,
        'lr':           0.1,
        'weight-decay': 5e-4,
        'dropout':      0.3,
    },
    metric_definitions=[
        {'Name': 'train:loss',  'Regex': r'train_loss=(\S+)'},
        {'Name': 'train:acc',   'Regex': r'train_acc=(\S+)%'},
        {'Name': 'val:loss',    'Regex': r'val_loss=(\S+)'},
        {'Name': 'val:acc',     'Regex': r'val_acc=(\S+)%'},
    ],
    keep_alive_period_in_seconds=300,  # warm pool for quick re-runs
)

print('Estimator configured.')

## 3. Launch the training job

In [ ]:
estimator.fit(
    inputs={'training': s3_data_uri},
    wait=True,      # stream logs to this notebook
    logs='All',
)

job_name = estimator.latest_training_job.name
print(f'\nJob name   : {job_name}')
print(f'Model S3   : {estimator.model_data}')

## 4. Plot training metrics from CloudWatch

In [ ]:
from sagemaker.analytics import TrainingJobAnalytics
import matplotlib.pyplot as plt

metrics_df = TrainingJobAnalytics(job_name).dataframe()
print(metrics_df.head())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for metric, ax, color in [
    ('val:acc',  axes[0], 'darkorange'),
    ('val:loss', axes[1], 'steelblue'),
]:
    sub = metrics_df[metrics_df['metric_name'] == metric]
    ax.plot(sub['timestamp'], sub['value'], color=color, linewidth=2)
    ax.set_title(metric)
    ax.set_xlabel('Time')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Download best checkpoint

In [ ]:
import tarfile, boto3

model_uri = estimator.model_data          # e.g. s3://bucket/output/model.tar.gz
bucket, key = model_uri.replace('s3://', '').split('/', 1)

local_tar = '../datasets/model.tar.gz'
boto3.client('s3', region_name=REGION).download_file(bucket, key, local_tar)

with tarfile.open(local_tar) as tar:
    tar.extractall('../datasets/trained_model/')

print('Model extracted to ../datasets/trained_model/')
import os; print(os.listdir('../datasets/trained_model/'))

---
**Next:** Open `03_deploy_endpoint.ipynb` to deploy the model as a real-time endpoint.